In [4]:
import os
import re
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pandas_gbq
from datetime import timedelta
from sqlalchemy import select, union_all, text
from inspectomop import Inspector
import bisect

# ENDPOINT in PHECODES
ENDPOINTS = {
    "obesity"              : ["278.1", "278.11"],
    "sleep_apnea"          : ["327.32", "327.3"],
    "gerd"                 : ["530.11"],
    "mdd"                  : ["296.22", "296.2"],
    "type2_diabetes"       : ["250.2"],
    "hypertension"         : ["401.1"],
    "hyperlipidemia"       : ["272.1", "272.11", "272.13"],
    "atrial_fibrillation"  : ["427.21"],
    "hf"                   : ["428.1","428.2","428.3", "428.4"], 
    "gad"                  : ["300.11"],
}
cdr = os.environ["WORKSPACE_CDR"]
bucket = os.getenv("WORKSPACE_BUCKET")

In [17]:
import os
import pandas_gbq
import pyarrow.parquet as pq

cdr = os.environ["WORKSPACE_CDR"]
bucket = os.getenv("WORKSPACE_BUCKET")

out_dir = "/home/jupyter/workspaces/controlled/ehr"
bucket_folder = os.path.join(bucket, "ehr")
os.makedirs(out_dir, exist_ok=True)
os.makedirs(bucket_folder, exist_ok=True)

tables_to_extract = {
    "condition_occurrence": "condition_occurrence.parquet",
    "drug_exposure": "drug.parquet",
    "procedure_occurrence": "procedure_occurrence.parquet",
    "visit_occurrence": "visit_occurrence.parquet",
    "person": "person.parquet",
}

for table, filename in tables_to_extract.items():
    pre_path = os.path.join(out_dir, filename)
    exclude_ids = "-1"  # 关键：默认值，防止 NameError

    if os.path.exists(pre_path):
        print(f"Loading previous data: {pre_path}")
        pre_df = pq.read_table(pre_path).to_pandas(ignore_metadata=True)
        if "person_id" in pre_df.columns and len(pre_df) > 0:
            existing_pids = pre_df["person_id"].dropna().astype("int64").unique().tolist()
            exclude_ids = ",".join(map(str, existing_pids)) if existing_pids else "-1"
    else:
        print(f"file not exists: {pre_path}")

    print(f"Pulling {table}...")
    query_sql = f"""
    WITH base AS (
      SELECT person_id
      FROM `{cdr}.cb_search_person`
      WHERE has_ehr_data = 1 AND has_fitbit = 1
        AND person_id NOT IN ({exclude_ids})
    )
    SELECT *
    FROM `{cdr}.{table}`
    WHERE person_id IN (SELECT person_id FROM base)
    """

    # 你之前要求要有费用上限，这里加上
    cfg = {"query": {"useLegacySql": False, "maximumBytesBilled": str(int(5e11))}}
    df = pandas_gbq.read_gbq(query_sql, project_id=cdr.split(".")[0], dialect="standard", configuration=cfg)

    out_path = os.path.join(out_dir, filename)
    gcs_path = os.path.join(bucket_folder, filename)

    df.to_parquet(out_path, index=False)
    df.to_parquet(gcs_path, index=False)

    print(f"✓ {table} saved → {out_path}")
    print(f"✓ {table} saved → {gcs_path}")


file not exists: /home/jupyter/workspaces/controlled/ehr/condition_occurrence.parquet
Pulling condition_occurrence...


GenericGBQException: Reason: 403 POST https://bigquery.googleapis.com/bigquery/v2/projects/fc-aou-cdr-prod-ct/jobs?prettyPrint=false: Access Denied: Project fc-aou-cdr-prod-ct: User does not have bigquery.jobs.create permission in project fc-aou-cdr-prod-ct.

Location: None
Job ID: 79d80174-1ac4-45da-81fb-ae1266cc1987


In [16]:
# load所有的ehr数据

import duckdb
from sqlalchemy import create_engine

out_dir = f"/home/jupyter/workspaces/controlled/all_ehr"
condition_path = os.path.join(out_dir, "condition_occurrence.parquet")
drug_path = os.path.join(out_dir, "drug.parquet")
procedure_path = os.path.join(out_dir, "procedure_occurrence.parquet")
visit_path = os.path.join(out_dir, "visit_occurrence.parquet")
person_path = os.path.join(out_dir, "person.parquet")

# Path to DuckDB file (persistent on disk)
duckdb_path = "/home/jupyter/workspaces/controlled/omop.duckdb"
con = duckdb.connect(duckdb_path)

engine = create_engine(f"duckdb:///{duckdb_path}")

print("DuckDB engine created:", engine)

def load_parquet(table, path):
    con.execute(f"DROP TABLE IF EXISTS {table}")
    con.execute(f"""
        CREATE TABLE {table} AS
        SELECT * FROM read_parquet('{path}')
    """)
    print(f"✓ Loaded {table}")

load_parquet("condition_occurrence", condition_path)
load_parquet("drug_exposure", drug_path)
load_parquet("procedure_occurrence", procedure_path)
load_parquet("visit_occurrence", visit_path)
load_parquet("person", person_path)
print("Closing raw data loading connection...")
con.close()

DuckDB engine created: Engine(duckdb:////home/jupyter/workspaces/controlled/omop.duckdb)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓ Loaded condition_occurrence


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓ Loaded drug_exposure


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓ Loaded procedure_occurrence


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓ Loaded visit_occurrence
✓ Loaded person
Closing raw data loading connection...


In [15]:
import os
import gc
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
from datetime import timedelta

# =============================================================================
# 1A. EHR COHORT CREATOR
# =============================================================================
def create_ehr_incident_cohort(disease_name, shared_pids, additional_concept_ids=None, additional_drug_concept_ids=None, min_days=90, verbose=True):
    base_cohort_dir = "/home/jupyter/workspaces/controlled/cohort_definitions"
    disease_dir = os.path.join(base_cohort_dir, disease_name)
    out_dir = "/home/jupyter/workspaces/controlled/all_ehr"
    os.makedirs(os.path.join(disease_dir, "parquet_labels"), exist_ok=True)
    
    label_out = os.path.join(disease_dir, "parquet_labels", f"label_{disease_name}_ehr.parquet")

    if verbose: print(f"--- CREATING EHR-ONLY COHORT: {disease_name.upper()} ---")

    # 1. Load EHR Events and compute global start dates
    cond_df = pq.read_table(os.path.join(out_dir, "condition_occurrence.parquet"), columns=["person_id", "condition_start_date", "condition_concept_id"]).to_pandas()
    cond_df["date"] = pd.to_datetime(cond_df["condition_start_date"], errors='coerce').dt.tz_localize(None)
    
    drug_df = pd.DataFrame(columns=["person_id", "date", "drug_concept_id"])
    if os.path.exists(os.path.join(out_dir, "drug.parquet")):
        drug_df = pq.read_table(os.path.join(out_dir, "drug.parquet"), columns=["person_id", "drug_exposure_start_date", "drug_concept_id"]).to_pandas()
        drug_df["date"] = pd.to_datetime(drug_df["drug_exposure_start_date"], errors='coerce').dt.tz_localize(None)

    all_dates = pd.concat([cond_df[["person_id", "date"]], drug_df[["person_id", "date"]]]).dropna()
    ehr_stats = all_dates.groupby("person_id")["date"].agg(['min', 'max']).rename(columns={"min": "ehr_start", "max": "ehr_last"})
    ehr_trigger_map = all_dates.groupby("person_id")["date"].apply(lambda x: sorted(list(set(x)))).to_dict()

    # 2. Identify Disease Dates
    d_cids = additional_concept_ids or []
    d_drugs = additional_drug_concept_ids or []
    dis_events = pd.concat([
        cond_df[cond_df["condition_concept_id"].isin(d_cids)][["person_id", "date"]],
        drug_df[drug_df["drug_concept_id"].isin(d_drugs)][["person_id", "date"]]
    ])
    disease_map = dis_events.groupby("person_id")["date"].apply(lambda x: sorted(list(set(x)))).to_dict()

    # 3. Generate Snapshots
    cohort_rows = []
    
    for pid in shared_pids:
        if pid not in ehr_stats.index: continue
        stats = ehr_stats.loc[pid]
        valid_start = stats["ehr_start"] + timedelta(days=min_days)
        triggers = ehr_trigger_map.get(pid, [])
        
        if pid in disease_map:
            d_dates = disease_map[pid]
            seen_dates = set() # Prevent duplicates from overlapping 3-month windows
            
            for d_date in d_dates:
                # 3-Month Window Centered on Disease (+/- 45 days)
                window_start = d_date - timedelta(days=90)
                window_end = d_date + timedelta(days=90)
                
                # A. Positive Case (Exactly 1 day prior)
                pos_time = d_date - timedelta(days=1)
                if pos_time >= valid_start and pos_time not in seen_dates:
                    cohort_rows.append({"person_id": pid, "prediction_time": pos_time, "boolean_value": True})
                    seen_dates.add(pos_time)
                
                # B. Negative Cases (All other triggers in the 3-month window)
                valid_triggers = [t for t in triggers if window_start <= t <= window_end and t >= valid_start]
                for t_date in valid_triggers:
                    if t_date != pos_time and t_date not in seen_dates:
                        cohort_rows.append({"person_id": pid, "prediction_time": t_date, "boolean_value": False})
                        seen_dates.add(t_date)
        else:
            # NEGATIVE PATIENT: 1 month prior to last record
            last_date = stats["ehr_last"]
            window_start = last_date - timedelta(days=90)
            window_end = last_date - timedelta(days=1)
            
            valid_triggers = [t for t in triggers if window_start <= t <= window_end and t >= valid_start]
            for t_date in valid_triggers:
                cohort_rows.append({"person_id": pid, "prediction_time": t_date, "boolean_value": False})

    label_df = pd.DataFrame(cohort_rows).drop_duplicates(subset=["person_id", "prediction_time"])
    label_df["prediction_time"] = label_df["prediction_time"].astype(str)
    pq.write_table(pa.Table.from_pandas(label_df), label_out)
    
    if verbose: print(f"Saved {len(label_df):,} EHR Snapshots (Pos: {label_df['boolean_value'].sum():,})\n")
    return label_out

# =============================================================================
# 1B. EHR SEQUENCE EXTRACTOR
# =============================================================================
def extract_ehr_sequences_batched(disease_name, max_seq_len=512, batch_size=1000):
    base_cohort_dir = "/home/jupyter/workspaces/controlled/cohort_definitions"
    label_path = os.path.join(base_cohort_dir, disease_name, "parquet_labels", f"label_{disease_name}_ehr.parquet")
    out_path = os.path.join(base_cohort_dir, disease_name, "inspectomop_features", f"transformer_{disease_name}_ehr.parquet")
    
    labels_all = pq.read_table(label_path).to_pandas()
    labels_all["index_date"] = pd.to_datetime(labels_all["prediction_time"])
    all_pids = labels_all["person_id"].unique()
    
    global_stats = {"total_rows": 0, "total_empty_ehr": 0, "total_empty_wearable": 0}
    
    writer = None
    for i in range(0, len(all_pids), batch_size):
        
        batch_pids = all_pids[i : i + batch_size].tolist()
        batch_labels = labels_all[labels_all["person_id"].isin(batch_pids)].copy()
        
        pid_str = ",".join(map(str, batch_pids))
        
        # 1. Fetch Clinical Events
        events_sql = f"""
            SELECT person_id, condition_concept_id AS concept_id, condition_start_date AS event_date
            FROM condition_occurrence WHERE person_id IN ({pid_str})
            UNION ALL
            SELECT person_id, drug_concept_id, drug_exposure_start_date
            FROM drug_exposure WHERE person_id IN ({pid_str})
            UNION ALL
            SELECT person_id, procedure_concept_id, procedure_date
            FROM procedure_occurrence WHERE person_id IN ({pid_str})
            UNION ALL
            SELECT person_id, visit_concept_id, visit_start_date
            FROM visit_occurrence WHERE person_id IN ({pid_str})
        """
        events_df = pd.read_sql(events_sql, engine)
        events_df["event_date"] = pd.to_datetime(events_df["event_date"], errors='coerce')
        
        merged = events_df.merge(batch_labels[["person_id", "index_date"]], on="person_id")
        merged["time_deltas"] = (merged["index_date"] - merged["event_date"]).dt.days
        merged = merged[merged["time_deltas"] > 0] # Strictly prior
        
        merged = merged.sort_values(["person_id", "index_date", "time_deltas"], ascending=[True, True, False])
        
        seqs = merged.groupby(["person_id", "index_date"]).agg({"concept_id": list, "time_deltas": list}).reset_index()
        seqs = seqs.rename(columns={"concept_id": "concept_ids"})
        
        # 2. Fetch and Inject Person (Demographics)
        if not seqs.empty:
            demo_sql = f"""
                SELECT person_id, gender_concept_id, race_concept_id, ethnicity_concept_id, birth_datetime
                FROM person WHERE person_id IN ({pid_str})
            """
            demo_df = pd.read_sql(demo_sql, engine)
            seqs = seqs.merge(demo_df, on="person_id", how="left")
            
            seqs["birth_datetime"] = pd.to_datetime(seqs["birth_datetime"], errors='coerce')
            index_naive = seqs["index_date"].dt.tz_localize(None)
            birth_naive = seqs["birth_datetime"].dt.tz_localize(None)
            
            seqs["age_at_index"] = (index_naive - birth_naive).dt.days // 365
            seqs["age_token"] = "AGE_" + seqs["age_at_index"].fillna(-1).astype(int).astype(str)
            
            def append_demographics(row):
                c_ids = row["concept_ids"]
                t_deltas = row["time_deltas"]
                demo_concepts = [
                    str(int(row["gender_concept_id"])) if pd.notna(row["gender_concept_id"]) else "UNK_GENDER", 
                    str(int(row["race_concept_id"])) if pd.notna(row["race_concept_id"]) else "UNK_RACE", 
                    str(int(row["ethnicity_concept_id"])) if pd.notna(row["ethnicity_concept_id"]) else "UNK_ETHNICITY",
                    row["age_token"]
                ]
                # Time delta is 0 for static demographics so they sit exactly at the prediction point
                return c_ids + demo_concepts, t_deltas + [0, 0, 0, 0]
                
            injected = seqs.apply(lambda x: pd.Series(append_demographics(x)), axis=1)
            seqs["concept_ids"] = injected[0]
            seqs["time_deltas"] = injected[1]
            
            # Clean up temporary demographic columns
            seqs = seqs.drop(columns=[
                "gender_concept_id", "race_concept_id", "ethnicity_concept_id", 
                "birth_datetime", "age_at_index", "age_token"
            ])
        
        # 3. Truncate
        seqs["concept_ids"] = seqs["concept_ids"].apply(lambda x: [str(c) for c in x][-max_seq_len:])
        seqs["time_deltas"] = seqs["time_deltas"].apply(lambda x: x[-max_seq_len:])
        
        table = pa.Table.from_pandas(seqs)
        if writer is None: writer = pq.ParquetWriter(out_path, table.schema)
        writer.write_table(table)
        
        ehr_lens = seqs["concept_ids"].apply(len)
        n_empty_ehr = (ehr_lens == 4).sum()
        
        global_stats['total_rows'] += len(seqs)
        
        print(f"Processed {len(seqs)} / {global_stats['total_rows']}, empty rows: {n_empty_ehr}")
        del events_df, merged, seqs
        gc.collect()
        
    if writer: writer.close()
    print(f"EHR Sequences saved to: {out_path}\n")
    return out_path

def create_shared_patient_pool_and_splits(disease_name, 
                                          additional_concept_ids=None, 
                                          additional_drug_concept_ids=None,
                                          min_history_days=90):
    """
    Finds patients who possess BOTH valid EHR and valid Wearable history,
    prints the exact counts, and generates a single shared split file.
    """
    out_dir = "/home/jupyter/workspaces/controlled/all_ehr"
    wear_path = "/home/jupyter/workspaces/controlled/wearable_daily_merged.parquet"
    
    print(f"\n{'='*80}")
    print(f"MASTER PATIENT INTERSECTION: {disease_name.upper()}")
    print(f"{'='*80}")
    
    # ---------------------------------------------------------
    # 1. Get Disease Dates (To check valid history prior to onset)
    # ---------------------------------------------------------
    cond_df = pq.read_table(os.path.join(out_dir, "condition_occurrence.parquet"), columns=["person_id", "condition_start_date", "condition_concept_id"]).to_pandas()
    cond_df["date"] = pd.to_datetime(cond_df["condition_start_date"], errors='coerce').dt.tz_localize(None)
    
    d_cids = additional_concept_ids or []
    dis_events = cond_df[cond_df["condition_concept_id"].isin(d_cids)][["person_id", "date"]]
    first_disease_map = dis_events.groupby("person_id")["date"].min().to_dict() # Earliest disease date
    
    # ---------------------------------------------------------
    # 2. Filter Valid EHR Patients
    # ---------------------------------------------------------
    ehr_stats = cond_df.groupby("person_id")["date"].agg(['min', 'max']).rename(columns={"min": "start", "max": "last"})
    
    valid_ehr_pids = set()
    for pid, row in ehr_stats.iterrows():
        if pid in first_disease_map:
            # Positive: Must have 90 days of EHR history BEFORE disease
            if (first_disease_map[pid] - row["start"]).days >= min_history_days:
                valid_ehr_pids.add(pid)
        else:
            # Negative: Must have 90 days of EHR history overall
            if (row["last"] - row["start"]).days >= min_history_days:
                valid_ehr_pids.add(pid)
                
    # ---------------------------------------------------------
    # 3. Filter Valid Wearable Patients
    # ---------------------------------------------------------
    wear_df = pq.read_table(wear_path, columns=["person_id", "day"]).to_pandas()
    wear_df["day"] = pd.to_datetime(wear_df["day"])
    wear_stats = wear_df.groupby("person_id")["day"].agg(['min', 'max']).rename(columns={"min": "start", "max": "last"})
    
    valid_wear_pids = set()
    for pid, row in wear_stats.iterrows():
        if pid in first_disease_map:
            # Positive: Must have min_history_days days of Wearable history BEFORE disease
            if (first_disease_map[pid] - row["start"]).days >= min_history_days:
                valid_wear_pids.add(pid)
        else:
            # Negative: Must have min_history_days days of Wearable history overall
            if (row["last"] - row["start"]).days >= min_history_days:
                valid_wear_pids.add(pid)

    # ---------------------------------------------------------
    # 4. INTERSECTION & PRINT STATS
    # ---------------------------------------------------------
    shared_pids = valid_ehr_pids.intersection(valid_wear_pids)
    
    pos_shared = len([p for p in shared_pids if p in first_disease_map])
    neg_shared = len(shared_pids) - pos_shared
    
    print(f"Total Valid EHR Patients:      {len(valid_ehr_pids):,}")
    print(f"Total Valid Wearable Patients: {len(valid_wear_pids):,}")
    print(f"--------------------------------------------------")
    print(f"SHARED PATIENTS (Intersection): {len(shared_pids):,}")
    print(f"  -> Positives (Will get disease): {pos_shared:,}")
    print(f"  -> Negatives (No disease):       {neg_shared:,}")
    print(f"{'='*80}\n")
    
    # ---------------------------------------------------------
    # 5. Create Master Shared Split
    # ---------------------------------------------------------
    base_cohort_dir = "/home/jupyter/workspaces/controlled/cohort_definitions"
    disease_dir = os.path.join(base_cohort_dir, disease_name)
    os.makedirs(os.path.join(disease_dir, "patient_splits"), exist_ok=True)
    split_out = os.path.join(disease_dir, "patient_splits", f"master_shared_split_{min_history_days}d.parquet")
    
    np.random.seed(42)
    pos_pids_arr = np.array([p for p in shared_pids if p in first_disease_map])
    neg_pids_arr = np.array([p for p in shared_pids if p not in first_disease_map])
    
    np.random.shuffle(pos_pids_arr)
    np.random.shuffle(neg_pids_arr)
    
    def get_split_labels(arr):
        n = len(arr); nt = int(n*0.2); nv = int(n*0.2)
        return set(arr[:nt]), set(arr[nt:nt+nv]), set(arr[nt+nv:])

    pt, pv, ptr = get_split_labels(pos_pids_arr)
    nt, nv, ntr = get_split_labels(neg_pids_arr)
    test, val = pt|nt, pv|nv
    
    split_data = [{"person_id": p, "split": "test" if p in test else "val" if p in val else "train"} for p in shared_pids]
    pq.write_table(pa.Table.from_pandas(pd.DataFrame(split_data)), split_out)
    
    print(f"Master Split saved to: {split_out}")
    return shared_pids

# =============================================================================
# 2A. WEARABLE COHORT CREATOR
# =============================================================================
def create_wearable_incident_cohort(disease_name, shared_pids, additional_concept_ids=None, min_days=90, verbose=True):
    base_cohort_dir = "/home/jupyter/workspaces/controlled/cohort_definitions"
    disease_dir = os.path.join(base_cohort_dir, disease_name)
    os.makedirs(os.path.join(disease_dir, "parquet_labels"), exist_ok=True)
    
    label_out = os.path.join(disease_dir, "parquet_labels", f"label_{disease_name}_wear_only.parquet")
    wear_path = "/home/jupyter/workspaces/controlled/wearable_daily_merged.parquet"

    if verbose: print(f"--- CREATING WEARABLE-ONLY COHORT: {disease_name.upper()} ---")

    # 1. Load Wearable triggers
    wear_df = pq.read_table(wear_path, columns=["person_id", "day"]).to_pandas()
    wear_df["day"] = pd.to_datetime(wear_df["day"])
    
    wear_stats = wear_df.groupby("person_id")["day"].agg(['min', 'max']).rename(columns={"min": "wear_start", "max": "wear_last"})
    wear_trigger_map = wear_df.groupby("person_id")["day"].apply(lambda x: sorted(list(set(x)))).to_dict()

    # 2. Get Disease Dates
    out_dir = "/home/jupyter/workspaces/controlled/all_ehr"
    cond_df = pq.read_table(os.path.join(out_dir, "condition_occurrence.parquet"), columns=["person_id", "condition_start_date", "condition_concept_id"]).to_pandas()
    cond_df["date"] = pd.to_datetime(cond_df["condition_start_date"], errors='coerce').dt.tz_localize(None)
    
    d_cids = additional_concept_ids or []
    dis_events = cond_df[cond_df["condition_concept_id"].isin(d_cids)][["person_id", "date"]]
    disease_map = dis_events.groupby("person_id")["date"].apply(lambda x: sorted(list(set(x)))).to_dict()

    # 3. Generate Snapshots
    cohort_rows = []
    
    for pid in shared_pids:
        if pid not in wear_stats.index: continue
        stats = wear_stats.loc[pid]
        valid_start = stats["wear_start"] + timedelta(days=min_days)
        triggers = wear_trigger_map.get(pid, [])
        
        if pid in disease_map:
            # POSITIVE PATIENT: Up to 3 disease occurrences
            d_dates = disease_map[pid]
            seen_dates = set() 
            
            for d_date in d_dates:
                # 3-Month Window Centered on Disease (+/- 45 days)
                window_start = d_date - timedelta(days=30)
                window_end = d_date + timedelta(days=30)
                
                # A. Positive Case
                pos_time = d_date - timedelta(days=1)
                if pos_time >= valid_start and pos_time not in seen_dates:
                    cohort_rows.append({"person_id": pid, "prediction_time": pos_time, "boolean_value": True})
                    seen_dates.add(pos_time)
                
                # B. Negative Cases
                valid_triggers = [t for t in triggers if window_start <= t <= window_end and t >= valid_start]
                for t_date in valid_triggers:
                    if t_date != pos_time and t_date not in seen_dates:
                        cohort_rows.append({"person_id": pid, "prediction_time": t_date, "boolean_value": False})
                        seen_dates.add(t_date)
        else:
            # NEGATIVE PATIENT: 1 month prior to last wearable sync
            last_date = stats["wear_last"]
            window_start = last_date - timedelta(days=30)
            window_end = last_date - timedelta(days=1)
            
            valid_triggers = [t for t in triggers if window_start <= t <= window_end and t >= valid_start]
            for t_date in valid_triggers:
                cohort_rows.append({"person_id": pid, "prediction_time": t_date, "boolean_value": False})

    label_df = pd.DataFrame(cohort_rows).drop_duplicates(subset=["person_id", "prediction_time"])
    label_df["prediction_time"] = label_df["prediction_time"].astype(str)
    pq.write_table(pa.Table.from_pandas(label_df), label_out)
    
    if verbose: print(f"Saved {len(label_df):,} Wearable Snapshots (Pos: {label_df['boolean_value'].sum():,})\n")
    return label_out

# =============================================================================
# 2B. WEARABLE SEQUENCE EXTRACTOR
# =============================================================================
def extract_wearable_sequences_batched(disease_name, max_seq_len=365, batch_size=2000):
    base_cohort_dir = "/home/jupyter/workspaces/controlled/cohort_definitions"
    label_path = os.path.join(base_cohort_dir, disease_name, "parquet_labels", f"label_{disease_name}_wear.parquet")
    out_path = os.path.join(base_cohort_dir, disease_name, "inspectomop_features", f"transformer_{disease_name}_wear.parquet")
    wear_path = "/home/jupyter/workspaces/controlled/wearable_daily_merged.parquet"
    
    labels_all = pq.read_table(label_path).to_pandas()
    labels_all["index_date"] = pd.to_datetime(labels_all["prediction_time"])
    all_pids = labels_all["person_id"].unique()
    
    writer = None
    total_rows = 0
    
    for i in range(0, len(all_pids), batch_size):
        batch_pids = all_pids[i : i + batch_size].tolist()
        batch_labels = labels_all[labels_all["person_id"].isin(batch_pids)].copy()
        
        wear_df = pq.read_table(wear_path, filters=[('person_id', 'in', batch_pids)]).to_pandas()
        wear_df["day"] = pd.to_datetime(wear_df["day"])
        
        merged = wear_df.merge(batch_labels[["person_id", "index_date"]], on="person_id")
        merged["wearable_time_deltas"] = (merged["index_date"] - merged["day"]).dt.days
        merged = merged[merged["wearable_time_deltas"] > 0]
        
        merged = merged.sort_values(["person_id", "index_date", "wearable_time_deltas"], ascending=[True, True, False])
        
        seqs = merged.groupby(["person_id", "index_date"]).agg({
            "wearable_time_deltas": list,
            "steps_day": list,
            "hr_avg_day": list,
            "sleep_total_day": list
        }).reset_index()
        
        # 2. Fetch and Inject Person (Demographics)
        if not seqs.empty:
            pid_str = ",".join(map(str, batch_pids))
            demo_sql = f"""
                SELECT person_id, gender_concept_id, race_concept_id, ethnicity_concept_id, birth_datetime
                FROM person WHERE person_id IN ({pid_str})
            """
            demo_df = pd.read_sql(demo_sql, engine)
            seqs = seqs.merge(demo_df, on="person_id", how="left")
            
            seqs["birth_datetime"] = pd.to_datetime(seqs["birth_datetime"], errors='coerce')
            index_naive = seqs["index_date"].dt.tz_localize(None)
            birth_naive = seqs["birth_datetime"].dt.tz_localize(None)
            
            # Calculate Age at prediction time
            seqs["age_at_index"] = (index_naive - birth_naive).dt.days // 365
            seqs["age_token"] = "AGE_" + seqs["age_at_index"].fillna(-1).astype(int).astype(str)
            
            def extract_demographics(row):
                demo_concepts = [
                    str(int(row["gender_concept_id"])) if pd.notna(row["gender_concept_id"]) else "UNK_GENDER", 
                    str(int(row["race_concept_id"])) if pd.notna(row["race_concept_id"]) else "UNK_RACE", 
                    str(int(row["ethnicity_concept_id"])) if pd.notna(row["ethnicity_concept_id"]) else "UNK_ETHNICITY",
                    row["age_token"]
                ]
                # Return the concepts and a 0 time delta for each
                return demo_concepts, [0, 0, 0, 0]
                
            injected = seqs.apply(lambda x: pd.Series(extract_demographics(x)), axis=1)
            seqs["demo_concept_ids"] = injected[0]
            seqs["demo_time_deltas"] = injected[1]
            
            # Clean up temporary demographic columns
            seqs = seqs.drop(columns=[
                "gender_concept_id", "race_concept_id", "ethnicity_concept_id", 
                "birth_datetime", "age_at_index", "age_token"
            ])
        else:
            # Fallback if batch is entirely empty
            seqs["demo_concept_ids"] = pd.Series(dtype=object)
            seqs["demo_time_deltas"] = pd.Series(dtype=object)
        
        wear_lens = seqs["wearable_time_deltas"].apply(len)
        n_empty_wear = (wear_lens == 4).sum()
        
        total_rows += len(seqs)
        
        print(f"Processed {len(seqs)} / {total_rows}, empty rows: {n_empty_wear}")
        
        # Truncate
        for col in ["wearable_time_deltas", "steps_day", "hr_avg_day", "sleep_total_day"]:
            seqs[col] = seqs[col].apply(lambda x: x[-max_seq_len:])
            
        table = pa.Table.from_pandas(seqs)
        if writer is None: writer = pq.ParquetWriter(out_path, table.schema)
        writer.write_table(table)
        
    if writer: writer.close()
    print(f"Wearable Sequences saved to: {out_path}\n")
    return out_path

In [10]:
disease_configs = {
    "atrial_fibrillation": {
        "additional_concept_ids": [313217],
        "additional_drug_concept_ids": [],
        "target_phecodes": ["427.21"]
    },
    "gad": {
        "additional_concept_ids": [442077, 434613, 4113821],
        "additional_drug_concept_ids": [739209],
        "target_phecodes": ["300.11"]
    },
    "hyperlipidemia": {
        "additional_concept_ids": [432867, 432827, 438720],
        "additional_drug_concept_ids": [],
        "target_phecodes": ["272.1", "272.11", "272.13"]
    },
    "hypertension": {
        "additional_concept_ids": [320128, 381290, 316866],
        "additional_drug_concept_ids": [],
        "target_phecodes": ["401.1"]
    },
    "mdd": {
        "additional_concept_ids": [440383, 4282096],
        "additional_drug_concept_ids": [739209],
        "target_phecodes": ["296.22", "296.2"]
    },
    "gerd": {
        "additional_concept_ids": [4144111, 318800],
        "additional_drug_concept_ids": [19019418],
        "target_phecodes": ["530.11"]
    },
    "type2_diabetes": {
        "additional_concept_ids": [4193704],
        "additional_drug_concept_ids": [1539407, 1503297],
        "target_phecodes": ["250.2"]
    },
    "sleep_apnea": {
        "additional_concept_ids": [442588, 313459],
        "additional_drug_concept_ids": [],
        "target_phecodes": ["327.32", "327.3"]
    },
    "obesity": {
        "additional_concept_ids": [433736, 434005],
        "additional_drug_concept_ids": [1539407],
        "target_phecodes": ["278.1", "278.11"]
    },
    "hf": {
        "additional_concept_ids": [319835],
        "additional_drug_concept_ids": [19075601, 19077546],
        "target_phecodes": ["428.1", "428.2", "428.3", "428.4"]
    }
}

print(f"{'='*60}")

results = []

for disease_name, config in disease_configs.items():
    print(f"Processing: {disease_name.upper()}...")
    
    try:
        concept_ids = disease_configs[disease_name]["additional_concept_ids"]

        # 1. Establish the strictly matched patient pool and create the Master Split
        shared_pids = create_shared_patient_pool_and_splits(
            disease_name=disease_name, 
            additional_concept_ids=concept_ids, 
            min_history_days=90
        )

        # 2. Feed the shared_pids into the EHR label generator
        ehr_label_path = create_ehr_incident_cohort(
            disease_name=disease_name, 
            shared_pids=shared_pids, 
            additional_concept_ids=concept_ids
        )

        # 3. Feed the exact same shared_pids into the Wearable label generator
        wear_label_path = create_wearable_incident_cohort(
            disease_name=disease_name, 
            shared_pids=shared_pids, 
            additional_concept_ids=concept_ids
        )

        # 4. Extract sequences (these automatically read the labels generated above)
        extract_ehr_sequences_batched(disease_name)
        extract_wearable_sequences_batched(disease_name, batch_size=500)
        print(f"  > Cohort created: {cohort_stats['pos']} Positives / {cohort_stats['neg']} Negatives")

        # InspectOMOP Features (Tabular)
#         extract_inspectomop_features_batched(disease_name, verbose=True)
#         print(f"  > InspectOMOP features extracted.")

        results.append({
            "Disease": disease_name,
            "Positives": cohort_stats['positive_cases'],
            "Negatives": cohort_stats['negative_cases'],
            "Status": "Success"
        })
        
    except Exception as e:
        print(f"  > ERROR: {str(e)}")
        results.append({
            "Disease": disease_name,
            "Positives": 0,
            "Negatives": 0,
            "Status": f"Failed: {str(e)}"
        })
        
    print("-" * 40)

# 3. Final Summary
# print("\n" + "="*60)
# print("BATCH PROCESSING COMPLETE")
# print("="*60)
# summary_df = pd.DataFrame(results)
# print(summary_df)

Processing: ATRIAL_FIBRILLATION...

MASTER PATIENT INTERSECTION: ATRIAL_FIBRILLATION
  > ERROR: data type 'dbdate' not understood
----------------------------------------
Processing: GAD...

MASTER PATIENT INTERSECTION: GAD
  > ERROR: data type 'dbdate' not understood
----------------------------------------
Processing: HYPERLIPIDEMIA...

MASTER PATIENT INTERSECTION: HYPERLIPIDEMIA
  > ERROR: data type 'dbdate' not understood
----------------------------------------
Processing: HYPERTENSION...

MASTER PATIENT INTERSECTION: HYPERTENSION
  > ERROR: data type 'dbdate' not understood
----------------------------------------
Processing: MDD...

MASTER PATIENT INTERSECTION: MDD
  > ERROR: data type 'dbdate' not understood
----------------------------------------
Processing: GERD...

MASTER PATIENT INTERSECTION: GERD
  > ERROR: data type 'dbdate' not understood
----------------------------------------
Processing: TYPE2_DIABETES...

MASTER PATIENT INTERSECTION: TYPE2_DIABETES
  > ERROR: data t

In [9]:
import os, shutil, subprocess

BASE = "/home/jupyter/workspaces/controlled"
BUCKET = os.environ["WORKSPACE_BUCKET"].rstrip("/")

src_dirs = [
    f"{BASE}/all_ehr",
    f"{BASE}/ehr",
    f"{BASE}/ehr_data",
]
dst_dir = f"{BASE}/all_ehr"
os.makedirs(dst_dir, exist_ok=True)

# 目标文件名 -> 可接受的同义源文件名（按顺序尝试）
needed = {
    "condition_occurrence.parquet": ["condition_occurrence.parquet", "condition.parquet"],
    "drug.parquet": ["drug.parquet", "drug_exposure.parquet"],
    "procedure_occurrence.parquet": ["procedure_occurrence.parquet"],
    "visit_occurrence.parquet": ["visit_occurrence.parquet"],
    "person.parquet": ["person.parquet"],
}

def copy_local(src, dst):
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy2(src, dst)

def copy_gcs(gcs_src, local_dst):
    cmd = f"gsutil cp '{gcs_src}' '{local_dst}'"
    p = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    return p.returncode == 0

for target, aliases in needed.items():
    dst = f"{dst_dir}/{target}"
    if os.path.exists(dst):
        print(f"[OK] {dst}")
        continue

    done = False
    # 1) 先找本地
    for d in src_dirs:
        for a in aliases:
            cand = f"{d}/{a}"
            if os.path.exists(cand):
                copy_local(cand, dst)
                print(f"[LOCAL COPY] {cand} -> {dst}")
                done = True
                break
        if done:
            break

    # 2) 再找 GCS
    if not done:
        for pref in ["ehr", "ehr_data", "data"]:
            for a in aliases:
                gcs = f"{BUCKET}/{pref}/{a}"
                if copy_gcs(gcs, dst):
                    print(f"[GCS COPY] {gcs} -> {dst}")
                    done = True
                    break
            if done:
                break

    if not done:
        print(f"[MISSING] {target}")

# wearable file（后续也会用到）
wear_local = f"{BASE}/wearable_daily_merged.parquet"
if not os.path.exists(wear_local):
    gcs_wear = f"{BUCKET}/wearable_daily_merged.parquet"
    ok = copy_gcs(gcs_wear, wear_local)
    print(f"[{'GCS COPY' if ok else 'MISSING'}] {gcs_wear} -> {wear_local}")

print("\nFinal check:")
for k in needed:
    p = f"{dst_dir}/{k}"
    print(f"{p}: {'OK' if os.path.exists(p) else 'MISSING'}")
print(f"{wear_local}: {'OK' if os.path.exists(wear_local) else 'MISSING'}")


[GCS COPY] gs://fc-secure-19f6cff6-9b8e-4550-b59f-058d7f345128/ehr/condition_occurrence.parquet -> /home/jupyter/workspaces/controlled/all_ehr/condition_occurrence.parquet
[GCS COPY] gs://fc-secure-19f6cff6-9b8e-4550-b59f-058d7f345128/ehr/drug.parquet -> /home/jupyter/workspaces/controlled/all_ehr/drug.parquet
[GCS COPY] gs://fc-secure-19f6cff6-9b8e-4550-b59f-058d7f345128/ehr/procedure_occurrence.parquet -> /home/jupyter/workspaces/controlled/all_ehr/procedure_occurrence.parquet
[GCS COPY] gs://fc-secure-19f6cff6-9b8e-4550-b59f-058d7f345128/ehr/visit_occurrence.parquet -> /home/jupyter/workspaces/controlled/all_ehr/visit_occurrence.parquet
[GCS COPY] gs://fc-secure-19f6cff6-9b8e-4550-b59f-058d7f345128/ehr/person.parquet -> /home/jupyter/workspaces/controlled/all_ehr/person.parquet
[GCS COPY] gs://fc-secure-19f6cff6-9b8e-4550-b59f-058d7f345128/wearable_daily_merged.parquet -> /home/jupyter/workspaces/controlled/wearable_daily_merged.parquet

Final check:
/home/jupyter/workspaces/control

In [2]:
import os
import pandas as pd
import pyarrow.parquet as pq

def audit_leakage_local(
    disease_name,
    endpoint_condition_cids,
    endpoint_drug_cids=None,
    base_dir="/home/jupyter/workspaces/controlled"
):
    endpoint_drug_cids = endpoint_drug_cids or []

    label_path = os.path.join(base_dir, "cohort_definitions", disease_name, "parquet_labels", f"label_{disease_name}_ehr.parquet")
    cond_path = os.path.join(base_dir, "all_ehr", "condition_occurrence.parquet")
    drug_path = os.path.join(base_dir, "all_ehr", "drug.parquet")

    labels = pq.read_table(label_path).to_pandas()
    labels["prediction_time"] = pd.to_datetime(labels["prediction_time"], errors="coerce")
    labels["boolean_value"] = labels["boolean_value"].astype(bool)

    cond = pq.read_table(cond_path, columns=["person_id", "condition_start_date", "condition_concept_id"]).to_pandas()
    cond["event_date"] = pd.to_datetime(cond["condition_start_date"], errors="coerce").dt.tz_localize(None)
    cond_ep = cond[cond["condition_concept_id"].isin(endpoint_condition_cids)][["person_id", "event_date"]]

    if os.path.exists(drug_path) and endpoint_drug_cids:
        drug = pq.read_table(drug_path, columns=["person_id", "drug_exposure_start_date", "drug_concept_id"]).to_pandas()
        drug["event_date"] = pd.to_datetime(drug["drug_exposure_start_date"], errors="coerce").dt.tz_localize(None)
        drug_ep = drug[drug["drug_concept_id"].isin(endpoint_drug_cids)][["person_id", "event_date"]]
    else:
        drug_ep = pd.DataFrame(columns=["person_id", "event_date"])

    ep = pd.concat([cond_ep, drug_ep], ignore_index=True).dropna()
    if ep.empty:
        print("No endpoint events found.")
        return

    first_ep = ep.groupby("person_id")["event_date"].min().rename("first_event").reset_index()
    L = labels.merge(first_ep, on="person_id", how="left")

    # snapshot 是否已在首次发病之后（应当避免出现在负样本）
    L["after_onset"] = L["first_event"].notna() & (L["prediction_time"] >= L["first_event"])

    # snapshot 在预测时点前是否已有 endpoint 事件（直接泄漏）
    tmp = L[["person_id", "prediction_time", "boolean_value"]].merge(ep, on="person_id", how="left")
    tmp = tmp[tmp["event_date"] < tmp["prediction_time"]].copy()
    tmp["days_before"] = (tmp["prediction_time"] - tmp["event_date"]).dt.days
    prior_flag = tmp.groupby(["person_id", "prediction_time"]).size().rename("n_prior_ep").reset_index()
    L = L.merge(prior_flag, on=["person_id", "prediction_time"], how="left")
    L["has_prior_ep"] = L["n_prior_ep"].fillna(0) > 0

    def pct(x): return round(100.0 * x.mean(), 2) if len(x) else 0.0

    print(f"\n=== Leakage audit: {disease_name} ===")
    print("n_snapshots:", len(L))
    print("n_pos:", int(L["boolean_value"].sum()), "n_neg:", int((~L["boolean_value"]).sum()))

    pos = L[L["boolean_value"]]
    neg = L[~L["boolean_value"]]

    print("\n[Risk-1] prior endpoint present before prediction_time")
    print("pos_has_prior_ep:", int(pos["has_prior_ep"].sum()), f"({pct(pos['has_prior_ep'])}%)")
    print("neg_has_prior_ep:", int(neg["has_prior_ep"].sum()), f"({pct(neg['has_prior_ep'])}%)")

    print("\n[Risk-2] prediction_time after first onset")
    print("pos_after_onset:", int(pos["after_onset"].sum()), f"({pct(pos['after_onset'])}%)")
    print("neg_after_onset:", int(neg["after_onset"].sum()), f"({pct(neg['after_onset'])}%)")

    if not tmp.empty:
        print("\n[How close leakage is] prior endpoint within 7/30 days")
        near7 = tmp.groupby(["person_id", "prediction_time"])["days_before"].min().le(7).mean()
        near30 = tmp.groupby(["person_id", "prediction_time"])["days_before"].min().le(30).mean()
        print("any_snapshot_min_gap<=7d:", round(near7*100, 2), "%")
        print("any_snapshot_min_gap<=30d:", round(near30*100, 2), "%")

# 示例（把concept id替换成你该病种配置）
# audit_leakage_local("hf", endpoint_condition_cids=[319835], endpoint_drug_cids=[19075601, 19077546])


In [12]:
import os
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

# =========================
# 0) 固定路径与疾病配置（可直接跑）
# =========================
ALL_EHR_DIR = "/home/jupyter/workspaces/controlled/all_ehr"
COHORT_DIR = "/home/jupyter/workspaces/controlled/cohort_definitions"
REPORT_CSV = "/home/jupyter/workspaces/controlled/report/ehr_label_leakage_audit.csv"

DISEASE_CONFIGS = {
    "obesity":             {"condition": [433736, 434005], "drug": [1539407]},
    "sleep_apnea":         {"condition": [442588, 313459], "drug": []},
    "gerd":                {"condition": [4144111, 318800], "drug": [19019418]},
    "mdd":                 {"condition": [440383, 4282096], "drug": [739209]},
    "type2_diabetes":      {"condition": [4193704], "drug": [1539407, 1503297]},
    "hypertension":        {"condition": [320128, 381290, 316866], "drug": []},
    "hyperlipidemia":      {"condition": [432867, 432827, 438720], "drug": []},
    "atrial_fibrillation": {"condition": [313217], "drug": []},
    "hf":                  {"condition": [319835], "drug": [19075601, 19077546]},
    "gad":                 {"condition": [442077, 434613, 4113821], "drug": [739209]},
}

# =========================
# 1) 读取工具（修复 dbdate metadata 问题）
# =========================
def read_parquet_safe(path, columns=None):
    table = pq.read_table(path, columns=columns)
    # 关键：忽略 parquet 中的 pandas metadata（避免 dbdate 报错）
    return table.to_pandas(ignore_metadata=True)

def to_naive_datetime(series):
    # 同时兼容 naive / tz-aware / date 类型
    return pd.to_datetime(series, errors="coerce", utc=True).dt.tz_convert(None)

# =========================
# 2) 读本地 EHR 事件表
# =========================
cond_path = os.path.join(ALL_EHR_DIR, "condition_occurrence.parquet")
drug_path = os.path.join(ALL_EHR_DIR, "drug.parquet")

if not os.path.exists(cond_path):
    raise FileNotFoundError(f"Missing: {cond_path}")

cond_df = read_parquet_safe(
    cond_path,
    columns=["person_id", "condition_concept_id", "condition_start_date"]
)
cond_df["event_date"] = to_naive_datetime(cond_df["condition_start_date"])
cond_df = cond_df.dropna(subset=["person_id", "condition_concept_id", "event_date"])
cond_df["person_id"] = pd.to_numeric(cond_df["person_id"], errors="coerce").astype("Int64")
cond_df["condition_concept_id"] = pd.to_numeric(cond_df["condition_concept_id"], errors="coerce").astype("Int64")
cond_df = cond_df.dropna(subset=["person_id", "condition_concept_id"]).copy()
cond_df["person_id"] = cond_df["person_id"].astype("int64")
cond_df["condition_concept_id"] = cond_df["condition_concept_id"].astype("int64")

if os.path.exists(drug_path):
    drug_df = read_parquet_safe(
        drug_path,
        columns=["person_id", "drug_concept_id", "drug_exposure_start_date"]
    )
    drug_df["event_date"] = to_naive_datetime(drug_df["drug_exposure_start_date"])
    drug_df = drug_df.dropna(subset=["person_id", "drug_concept_id", "event_date"])
    drug_df["person_id"] = pd.to_numeric(drug_df["person_id"], errors="coerce").astype("Int64")
    drug_df["drug_concept_id"] = pd.to_numeric(drug_df["drug_concept_id"], errors="coerce").astype("Int64")
    drug_df = drug_df.dropna(subset=["person_id", "drug_concept_id"]).copy()
    drug_df["person_id"] = drug_df["person_id"].astype("int64")
    drug_df["drug_concept_id"] = drug_df["drug_concept_id"].astype("int64")
else:
    drug_df = pd.DataFrame(columns=["person_id", "drug_concept_id", "event_date"])

print("Loaded condition rows:", len(cond_df))
print("Loaded drug rows:", len(drug_df))

# =========================
# 3) 工具函数
# =========================
def resolve_label_file(disease_name):
    candidates = [
        os.path.join(COHORT_DIR, disease_name, "parquet_labels", f"label_{disease_name}_ehr.parquet"),
        os.path.join(COHORT_DIR, disease_name, "parquet_labels", f"label_{disease_name}_90d.parquet"),
        os.path.join(COHORT_DIR, disease_name, "parquet_labels", f"label_{disease_name}.parquet"),
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    return None

def normalize_labels(df):
    # person_id
    if "person_id" not in df.columns and "subject_id" in df.columns:
        df = df.rename(columns={"subject_id": "person_id"})
    if "person_id" not in df.columns:
        raise ValueError("Label file missing person_id/subject_id")

    # prediction_time
    if "prediction_time" not in df.columns and "index_date" in df.columns:
        df = df.rename(columns={"index_date": "prediction_time"})
    if "prediction_time" not in df.columns:
        raise ValueError("Label file missing prediction_time/index_date")

    # label column
    label_col = None
    for c in ["boolean_value", "boolean", "label", "outcome", "target"]:
        if c in df.columns:
            label_col = c
            break
    if label_col is None:
        bool_cols = df.select_dtypes(include=["bool"]).columns.tolist()
        if bool_cols:
            label_col = bool_cols[0]
        else:
            raise ValueError("No label column found")

    out = df[["person_id", "prediction_time", label_col]].copy()
    out = out.rename(columns={label_col: "y"})
    out["person_id"] = pd.to_numeric(out["person_id"], errors="coerce").astype("Int64")
    out["prediction_time"] = to_naive_datetime(out["prediction_time"])

    # y -> 0/1
    if out["y"].dtype == bool:
        out["y"] = out["y"].astype(int)
    else:
        if out["y"].dtype == object:
            m = out["y"].astype(str).str.strip().str.lower().map({
                "true": 1, "false": 0, "1": 1, "0": 0
            })
            out["y"] = m.where(m.notna(), pd.to_numeric(out["y"], errors="coerce"))
        else:
            out["y"] = pd.to_numeric(out["y"], errors="coerce")
        out["y"] = out["y"].fillna(0).astype(int).clip(0, 1)

    out = out.dropna(subset=["person_id", "prediction_time"]).copy()
    out["person_id"] = out["person_id"].astype("int64")
    out = out.reset_index(drop=True)
    return out

def build_endpoint_events(cfg):
    cond_ep = cond_df[cond_df["condition_concept_id"].isin(cfg["condition"])][["person_id", "event_date"]]
    if len(cfg["drug"]) > 0 and len(drug_df) > 0:
        drug_ep = drug_df[drug_df["drug_concept_id"].isin(cfg["drug"])][["person_id", "event_date"]]
    else:
        drug_ep = pd.DataFrame(columns=["person_id", "event_date"])
    ep = pd.concat([cond_ep, drug_ep], ignore_index=True).dropna(subset=["person_id", "event_date"])
    return ep

def audit_one_disease(disease_name, cfg):
    label_file = resolve_label_file(disease_name)
    if label_file is None:
        return {
            "disease": disease_name,
            "status": "missing_label_file",
            "label_file": None
        }

    try:
        raw = read_parquet_safe(label_file)
        L = normalize_labels(raw)
    except Exception as e:
        return {
            "disease": disease_name,
            "status": f"label_parse_error: {str(e)}",
            "label_file": label_file
        }

    ep = build_endpoint_events(cfg)
    if ep.empty or L.empty:
        return {
            "disease": disease_name,
            "status": "no_events_or_labels",
            "label_file": label_file,
            "n_snapshots": int(len(L)),
            "n_endpoint_events": int(len(ep))
        }

    first_event = ep.groupby("person_id")["event_date"].min().rename("first_event")
    L = L.merge(first_event, on="person_id", how="left")
    L["after_onset"] = L["first_event"].notna() & (L["prediction_time"] >= L["first_event"])

    event_map = {}
    for pid, g in ep.groupby("person_id"):
        arr = np.sort(g["event_date"].values.astype("datetime64[ns]"))
        event_map[int(pid)] = np.unique(arr)

    has_prior = np.zeros(len(L), dtype=bool)
    min_gap_days = np.full(len(L), np.nan)

    idx_map = L.groupby("person_id").indices
    pred_vals = L["prediction_time"].values.astype("datetime64[ns]")

    for pid, pos_idx in idx_map.items():
        ev = event_map.get(int(pid))
        if ev is None or len(ev) == 0:
            continue
        p = pred_vals[pos_idx]
        j = np.searchsorted(ev, p, side="left") - 1
        valid = j >= 0
        if valid.any():
            hit_idx = pos_idx[valid]
            has_prior[hit_idx] = True
            gaps = ((p[valid] - ev[j[valid]]) / np.timedelta64(1, "D")).astype(float)
            min_gap_days[hit_idx] = gaps

    L["has_prior_endpoint"] = has_prior
    L["min_gap_days"] = min_gap_days

    pos = L[L["y"] == 1]
    neg = L[L["y"] == 0]

    def pct(num, den):
        return round(100.0 * num / den, 2) if den > 0 else np.nan

    pos_prior_n = int(pos["has_prior_endpoint"].sum())
    neg_prior_n = int(neg["has_prior_endpoint"].sum())
    pos_after_n = int(pos["after_onset"].sum())
    neg_after_n = int(neg["after_onset"].sum())

    return {
        "disease": disease_name,
        "status": "ok",
        "label_file": label_file,
        "n_snapshots": int(len(L)),
        "n_pos": int((L["y"] == 1).sum()),
        "n_neg": int((L["y"] == 0).sum()),
        "n_endpoint_events": int(len(ep)),
        "n_endpoint_persons": int(ep["person_id"].nunique()),
        "pos_has_prior_endpoint_n": pos_prior_n,
        "pos_has_prior_endpoint_pct": pct(pos_prior_n, len(pos)),
        "neg_has_prior_endpoint_n": neg_prior_n,
        "neg_has_prior_endpoint_pct": pct(neg_prior_n, len(neg)),
        "pos_after_onset_n": pos_after_n,
        "pos_after_onset_pct": pct(pos_after_n, len(pos)),
        "neg_after_onset_n": neg_after_n,
        "neg_after_onset_pct": pct(neg_after_n, len(neg)),
        "pos_median_min_gap_days": float(np.nanmedian(pos["min_gap_days"])) if pos["has_prior_endpoint"].any() else np.nan,
        "neg_median_min_gap_days": float(np.nanmedian(neg["min_gap_days"])) if neg["has_prior_endpoint"].any() else np.nan,
    }

# =========================
# 4) 执行全部 disease
# =========================
results = []
for d, cfg in DISEASE_CONFIGS.items():
    print(f"Auditing: {d}")
    results.append(audit_one_disease(d, cfg))

audit_df = pd.DataFrame(results)

if "status" in audit_df.columns:
    ok_df = audit_df[audit_df["status"] == "ok"].copy()
    if not ok_df.empty:
        ok_df = ok_df.sort_values(
            ["neg_has_prior_endpoint_pct", "neg_after_onset_pct", "pos_has_prior_endpoint_pct"],
            ascending=[False, False, False]
        )
        display(ok_df)
    else:
        display(audit_df)
else:
    display(audit_df)

os.makedirs(os.path.dirname(REPORT_CSV), exist_ok=True)
audit_df.to_csv(REPORT_CSV, index=False)
print(f"\nSaved audit report to: {REPORT_CSV}")


Loaded condition rows: 12749377
Loaded drug rows: 7487105
Auditing: obesity
Auditing: sleep_apnea
Auditing: gerd
Auditing: mdd
Auditing: type2_diabetes
Auditing: hypertension
Auditing: hyperlipidemia
Auditing: atrial_fibrillation
Auditing: hf
Auditing: gad


,disease,status,label_file
0,obesity,missing_label_file,None
1,sleep_apnea,missing_label_file,None
2,gerd,missing_label_file,None
3,mdd,missing_label_file,None
4,type2_diabetes,missing_label_file,None
5,hypertension,missing_label_file,None
6,hyperlipidemia,missing_label_file,None
7,atrial_fibrillation,missing_label_file,None
8,hf,missing_label_file,None
9,gad,missing_label_file,None



Saved audit report to: /home/jupyter/workspaces/controlled/report/ehr_label_leakage_audit.csv


In [18]:
import os
import gc
import shutil
import subprocess
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import duckdb
from datetime import timedelta
from sqlalchemy import create_engine

# =========================================================
# 0) 基础路径
# =========================================================
BASE = "/home/jupyter/workspaces/controlled"
ALL_EHR_DIR = f"{BASE}/all_ehr"
EHR_DIR = f"{BASE}/ehr"
EHR_DATA_DIR = f"{BASE}/ehr_data"
WEAR_PATH = f"{BASE}/wearable_daily_merged.parquet"
COHORT_BASE = f"{BASE}/cohort_definitions"
REPORT_DIR = f"{BASE}/report"
os.makedirs(REPORT_DIR, exist_ok=True)

BUCKET = os.environ.get("WORKSPACE_BUCKET", "").rstrip("/")

# =========================================================
# 1) 通用工具
# =========================================================
def read_parquet_safe(path, columns=None, filters=None):
    t = pq.read_table(path, columns=columns, filters=filters)
    return t.to_pandas(ignore_metadata=True)

def to_naive_datetime(s):
    return pd.to_datetime(s, errors="coerce", utc=True).dt.tz_convert(None)

def copy_from_gcs(gcs_src, local_dst):
    cmd = f"gsutil cp '{gcs_src}' '{local_dst}'"
    p = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    return p.returncode == 0

# =========================================================
# 2) 确保 all_ehr + wearable 文件存在
# =========================================================
os.makedirs(ALL_EHR_DIR, exist_ok=True)

needed = {
    "condition_occurrence.parquet": ["condition_occurrence.parquet", "condition.parquet"],
    "drug.parquet": ["drug.parquet", "drug_exposure.parquet"],
    "procedure_occurrence.parquet": ["procedure_occurrence.parquet"],
    "visit_occurrence.parquet": ["visit_occurrence.parquet"],
    "person.parquet": ["person.parquet"],
}

src_dirs = [ALL_EHR_DIR, EHR_DIR, EHR_DATA_DIR]

for target, aliases in needed.items():
    dst = f"{ALL_EHR_DIR}/{target}"
    if os.path.exists(dst):
        continue

    done = False
    # local copy
    for d in src_dirs:
        for a in aliases:
            cand = f"{d}/{a}"
            if os.path.exists(cand):
                shutil.copy2(cand, dst)
                print(f"[LOCAL COPY] {cand} -> {dst}")
                done = True
                break
        if done:
            break

    # gcs copy
    if not done and BUCKET:
        for pref in ["ehr", "ehr_data"]:
            for a in aliases:
                gcs = f"{BUCKET}/{pref}/{a}"
                if copy_from_gcs(gcs, dst):
                    print(f"[GCS COPY] {gcs} -> {dst}")
                    done = True
                    break
            if done:
                break

    if not done:
        raise FileNotFoundError(f"Missing required file: {dst}")

if not os.path.exists(WEAR_PATH) and BUCKET:
    if copy_from_gcs(f"{BUCKET}/wearable_daily_merged.parquet", WEAR_PATH):
        print(f"[GCS COPY] {BUCKET}/wearable_daily_merged.parquet -> {WEAR_PATH}")

if not os.path.exists(WEAR_PATH):
    raise FileNotFoundError(f"Missing wearable file: {WEAR_PATH}")

print("Input files ready.")

# =========================================================
# 3) DuckDB 装载
# =========================================================
duckdb_path = f"{BASE}/omop.duckdb"
con = duckdb.connect(duckdb_path)
engine = create_engine(f"duckdb:///{duckdb_path}")
print("DuckDB engine created:", engine)

def load_parquet_to_duckdb(table, path):
    con.execute(f"DROP TABLE IF EXISTS {table}")
    con.execute(f"CREATE TABLE {table} AS SELECT * FROM read_parquet('{path}')")
    print(f"✓ Loaded {table}")

load_parquet_to_duckdb("condition_occurrence", f"{ALL_EHR_DIR}/condition_occurrence.parquet")
load_parquet_to_duckdb("drug_exposure", f"{ALL_EHR_DIR}/drug.parquet")
load_parquet_to_duckdb("procedure_occurrence", f"{ALL_EHR_DIR}/procedure_occurrence.parquet")
load_parquet_to_duckdb("visit_occurrence", f"{ALL_EHR_DIR}/visit_occurrence.parquet")
load_parquet_to_duckdb("person", f"{ALL_EHR_DIR}/person.parquet")
con.close()
print("DuckDB loading done.")

# =========================================================
# 4) Cohort + Sequence 函数
# =========================================================
def create_shared_patient_pool_and_splits(
    disease_name,
    additional_concept_ids=None,
    additional_drug_concept_ids=None,
    min_history_days=90,
):
    cond_df = read_parquet_safe(
        f"{ALL_EHR_DIR}/condition_occurrence.parquet",
        columns=["person_id", "condition_start_date", "condition_concept_id"]
    )
    cond_df["date"] = to_naive_datetime(cond_df["condition_start_date"])
    cond_df = cond_df.dropna(subset=["person_id", "date"])
    cond_df["person_id"] = pd.to_numeric(cond_df["person_id"], errors="coerce").astype("Int64")
    cond_df["condition_concept_id"] = pd.to_numeric(cond_df["condition_concept_id"], errors="coerce").astype("Int64")
    cond_df = cond_df.dropna(subset=["person_id", "condition_concept_id"])
    cond_df["person_id"] = cond_df["person_id"].astype("int64")
    cond_df["condition_concept_id"] = cond_df["condition_concept_id"].astype("int64")

    drug_df = read_parquet_safe(
        f"{ALL_EHR_DIR}/drug.parquet",
        columns=["person_id", "drug_exposure_start_date", "drug_concept_id"]
    )
    drug_df["date"] = to_naive_datetime(drug_df["drug_exposure_start_date"])
    drug_df = drug_df.dropna(subset=["person_id", "date"])
    drug_df["person_id"] = pd.to_numeric(drug_df["person_id"], errors="coerce").astype("Int64")
    drug_df["drug_concept_id"] = pd.to_numeric(drug_df["drug_concept_id"], errors="coerce").astype("Int64")
    drug_df = drug_df.dropna(subset=["person_id", "drug_concept_id"])
    drug_df["person_id"] = drug_df["person_id"].astype("int64")
    drug_df["drug_concept_id"] = drug_df["drug_concept_id"].astype("int64")

    d_cids = set(additional_concept_ids or [])
    d_drugs = set(additional_drug_concept_ids or [])

    dis_cond = cond_df[cond_df["condition_concept_id"].isin(d_cids)][["person_id", "date"]]
    dis_drug = drug_df[drug_df["drug_concept_id"].isin(d_drugs)][["person_id", "date"]]
    dis_events = pd.concat([dis_cond, dis_drug], ignore_index=True)
    first_disease_map = dis_events.groupby("person_id")["date"].min().to_dict()

    all_dates = pd.concat([cond_df[["person_id", "date"]], drug_df[["person_id", "date"]]], ignore_index=True).dropna()
    ehr_stats = all_dates.groupby("person_id")["date"].agg(["min", "max"]).rename(columns={"min": "start", "max": "last"})

    valid_ehr = set()
    for pid, row in ehr_stats.iterrows():
        if pid in first_disease_map:
            if (first_disease_map[pid] - row["start"]).days >= min_history_days:
                valid_ehr.add(pid)
        else:
            if (row["last"] - row["start"]).days >= min_history_days:
                valid_ehr.add(pid)

    wear_df = read_parquet_safe(WEAR_PATH, columns=["person_id", "day"])
    wear_df["day"] = to_naive_datetime(wear_df["day"])
    wear_df = wear_df.dropna(subset=["person_id", "day"])
    wear_df["person_id"] = pd.to_numeric(wear_df["person_id"], errors="coerce").astype("Int64")
    wear_df = wear_df.dropna(subset=["person_id"])
    wear_df["person_id"] = wear_df["person_id"].astype("int64")

    wear_stats = wear_df.groupby("person_id")["day"].agg(["min", "max"]).rename(columns={"min": "start", "max": "last"})
    valid_wear = set()
    for pid, row in wear_stats.iterrows():
        if pid in first_disease_map:
            if (first_disease_map[pid] - row["start"]).days >= min_history_days:
                valid_wear.add(pid)
        else:
            if (row["last"] - row["start"]).days >= min_history_days:
                valid_wear.add(pid)

    shared_pids = valid_ehr.intersection(valid_wear)

    disease_dir = f"{COHORT_BASE}/{disease_name}"
    os.makedirs(f"{disease_dir}/patient_splits", exist_ok=True)
    split_out = f"{disease_dir}/patient_splits/master_shared_split_{min_history_days}d.parquet"

    np.random.seed(42)
    pos = np.array([p for p in shared_pids if p in first_disease_map])
    neg = np.array([p for p in shared_pids if p not in first_disease_map])
    np.random.shuffle(pos)
    np.random.shuffle(neg)

    def split_ids(arr):
        n = len(arr)
        nt = int(n * 0.2)
        nv = int(n * 0.2)
        return set(arr[:nt]), set(arr[nt:nt + nv]), set(arr[nt + nv:])

    pt, pv, _ = split_ids(pos)
    nt, nv, _ = split_ids(neg)
    test = pt | nt
    val = pv | nv

    split_rows = [{"person_id": p, "split": ("test" if p in test else "val" if p in val else "train")} for p in shared_pids]
    pq.write_table(pa.Table.from_pandas(pd.DataFrame(split_rows), preserve_index=False), split_out)

    print(f"[{disease_name}] shared={len(shared_pids):,} split={split_out}")
    return shared_pids

def create_ehr_incident_cohort(disease_name, shared_pids, additional_concept_ids=None, additional_drug_concept_ids=None, min_days=90):
    disease_dir = f"{COHORT_BASE}/{disease_name}"
    os.makedirs(f"{disease_dir}/parquet_labels", exist_ok=True)
    label_out = f"{disease_dir}/parquet_labels/label_{disease_name}_ehr.parquet"

    cond_df = read_parquet_safe(
        f"{ALL_EHR_DIR}/condition_occurrence.parquet",
        columns=["person_id", "condition_start_date", "condition_concept_id"]
    )
    cond_df["date"] = to_naive_datetime(cond_df["condition_start_date"])
    cond_df = cond_df.dropna(subset=["person_id", "date"])
    cond_df["person_id"] = pd.to_numeric(cond_df["person_id"], errors="coerce").astype("Int64")
    cond_df["condition_concept_id"] = pd.to_numeric(cond_df["condition_concept_id"], errors="coerce").astype("Int64")
    cond_df = cond_df.dropna(subset=["person_id", "condition_concept_id"])
    cond_df["person_id"] = cond_df["person_id"].astype("int64")
    cond_df["condition_concept_id"] = cond_df["condition_concept_id"].astype("int64")

    drug_df = read_parquet_safe(
        f"{ALL_EHR_DIR}/drug.parquet",
        columns=["person_id", "drug_exposure_start_date", "drug_concept_id"]
    )
    drug_df["date"] = to_naive_datetime(drug_df["drug_exposure_start_date"])
    drug_df = drug_df.dropna(subset=["person_id", "date"])
    drug_df["person_id"] = pd.to_numeric(drug_df["person_id"], errors="coerce").astype("Int64")
    drug_df["drug_concept_id"] = pd.to_numeric(drug_df["drug_concept_id"], errors="coerce").astype("Int64")
    drug_df = drug_df.dropna(subset=["person_id", "drug_concept_id"])
    drug_df["person_id"] = drug_df["person_id"].astype("int64")
    drug_df["drug_concept_id"] = drug_df["drug_concept_id"].astype("int64")

    all_dates = pd.concat([cond_df[["person_id", "date"]], drug_df[["person_id", "date"]]], ignore_index=True).dropna()
    ehr_stats = all_dates.groupby("person_id")["date"].agg(["min", "max"]).rename(columns={"min": "ehr_start", "max": "ehr_last"})
    ehr_trigger_map = all_dates.groupby("person_id")["date"].apply(lambda x: sorted(set(x))).to_dict()

    d_cids = set(additional_concept_ids or [])
    d_drugs = set(additional_drug_concept_ids or [])
    dis_events = pd.concat([
        cond_df[cond_df["condition_concept_id"].isin(d_cids)][["person_id", "date"]],
        drug_df[drug_df["drug_concept_id"].isin(d_drugs)][["person_id", "date"]],
    ], ignore_index=True)
    disease_map = dis_events.groupby("person_id")["date"].apply(lambda x: sorted(set(x))).to_dict()

    rows = []
    for pid in shared_pids:
        if pid not in ehr_stats.index:
            continue
        stats = ehr_stats.loc[pid]
        valid_start = stats["ehr_start"] + timedelta(days=min_days)
        triggers = ehr_trigger_map.get(pid, [])

        if pid in disease_map:
            seen = set()
            for d_date in disease_map[pid]:
                win_s = d_date - timedelta(days=90)
                win_e = d_date + timedelta(days=90)
                pos_time = d_date - timedelta(days=1)

                if pos_time >= valid_start and pos_time not in seen:
                    rows.append({"person_id": pid, "prediction_time": pos_time, "boolean_value": True})
                    seen.add(pos_time)

                valid_triggers = [t for t in triggers if win_s <= t <= win_e and t >= valid_start]
                for t_date in valid_triggers:
                    if t_date != pos_time and t_date not in seen:
                        rows.append({"person_id": pid, "prediction_time": t_date, "boolean_value": False})
                        seen.add(t_date)
        else:
            last_date = stats["ehr_last"]
            win_s = last_date - timedelta(days=90)
            win_e = last_date - timedelta(days=1)
            valid_triggers = [t for t in triggers if win_s <= t <= win_e and t >= valid_start]
            for t_date in valid_triggers:
                rows.append({"person_id": pid, "prediction_time": t_date, "boolean_value": False})

    label_df = pd.DataFrame(rows).drop_duplicates(subset=["person_id", "prediction_time"])
    label_df["prediction_time"] = pd.to_datetime(label_df["prediction_time"], errors="coerce").astype(str)
    pq.write_table(pa.Table.from_pandas(label_df, preserve_index=False), label_out)

    pos_n = int(label_df["boolean_value"].sum()) if not label_df.empty else 0
    neg_n = int((~label_df["boolean_value"]).sum()) if not label_df.empty else 0
    print(f"[{disease_name}] EHR label rows={len(label_df):,} pos={pos_n:,} neg={neg_n:,}")
    return label_out, {"positive_cases": pos_n, "negative_cases": neg_n}

def create_wearable_incident_cohort(disease_name, shared_pids, additional_concept_ids=None, min_days=90):
    disease_dir = f"{COHORT_BASE}/{disease_name}"
    os.makedirs(f"{disease_dir}/parquet_labels", exist_ok=True)
    label_out = f"{disease_dir}/parquet_labels/label_{disease_name}_wear_only.parquet"

    wear_df = read_parquet_safe(WEAR_PATH, columns=["person_id", "day"])
    wear_df["day"] = to_naive_datetime(wear_df["day"])
    wear_df = wear_df.dropna(subset=["person_id", "day"])
    wear_df["person_id"] = pd.to_numeric(wear_df["person_id"], errors="coerce").astype("Int64")
    wear_df = wear_df.dropna(subset=["person_id"])
    wear_df["person_id"] = wear_df["person_id"].astype("int64")

    wear_stats = wear_df.groupby("person_id")["day"].agg(["min", "max"]).rename(columns={"min": "wear_start", "max": "wear_last"})
    wear_trigger_map = wear_df.groupby("person_id")["day"].apply(lambda x: sorted(set(x))).to_dict()

    cond_df = read_parquet_safe(
        f"{ALL_EHR_DIR}/condition_occurrence.parquet",
        columns=["person_id", "condition_start_date", "condition_concept_id"]
    )
    cond_df["date"] = to_naive_datetime(cond_df["condition_start_date"])
    cond_df = cond_df.dropna(subset=["person_id", "date"])
    cond_df["person_id"] = pd.to_numeric(cond_df["person_id"], errors="coerce").astype("Int64")
    cond_df["condition_concept_id"] = pd.to_numeric(cond_df["condition_concept_id"], errors="coerce").astype("Int64")
    cond_df = cond_df.dropna(subset=["person_id", "condition_concept_id"])
    cond_df["person_id"] = cond_df["person_id"].astype("int64")
    cond_df["condition_concept_id"] = cond_df["condition_concept_id"].astype("int64")

    d_cids = set(additional_concept_ids or [])
    dis_events = cond_df[cond_df["condition_concept_id"].isin(d_cids)][["person_id", "date"]]
    disease_map = dis_events.groupby("person_id")["date"].apply(lambda x: sorted(set(x))).to_dict()

    rows = []
    for pid in shared_pids:
        if pid not in wear_stats.index:
            continue
        stats = wear_stats.loc[pid]
        valid_start = stats["wear_start"] + timedelta(days=min_days)
        triggers = wear_trigger_map.get(pid, [])

        if pid in disease_map:
            seen = set()
            for d_date in disease_map[pid]:
                win_s = d_date - timedelta(days=30)
                win_e = d_date + timedelta(days=30)
                pos_time = d_date - timedelta(days=1)

                if pos_time >= valid_start and pos_time not in seen:
                    rows.append({"person_id": pid, "prediction_time": pos_time, "boolean_value": True})
                    seen.add(pos_time)

                valid_triggers = [t for t in triggers if win_s <= t <= win_e and t >= valid_start]
                for t_date in valid_triggers:
                    if t_date != pos_time and t_date not in seen:
                        rows.append({"person_id": pid, "prediction_time": t_date, "boolean_value": False})
                        seen.add(t_date)
        else:
            last_date = stats["wear_last"]
            win_s = last_date - timedelta(days=30)
            win_e = last_date - timedelta(days=1)
            valid_triggers = [t for t in triggers if win_s <= t <= win_e and t >= valid_start]
            for t_date in valid_triggers:
                rows.append({"person_id": pid, "prediction_time": t_date, "boolean_value": False})

    label_df = pd.DataFrame(rows).drop_duplicates(subset=["person_id", "prediction_time"])
    label_df["prediction_time"] = pd.to_datetime(label_df["prediction_time"], errors="coerce").astype(str)
    pq.write_table(pa.Table.from_pandas(label_df, preserve_index=False), label_out)

    pos_n = int(label_df["boolean_value"].sum()) if not label_df.empty else 0
    neg_n = int((~label_df["boolean_value"]).sum()) if not label_df.empty else 0
    print(f"[{disease_name}] WEAR label rows={len(label_df):,} pos={pos_n:,} neg={neg_n:,}")
    return label_out, {"positive_cases": pos_n, "negative_cases": neg_n}

def extract_ehr_sequences_batched(disease_name, max_seq_len=512, batch_size=1000):
    label_path = f"{COHORT_BASE}/{disease_name}/parquet_labels/label_{disease_name}_ehr.parquet"
    out_path = f"{COHORT_BASE}/{disease_name}/inspectomop_features/transformer_{disease_name}_ehr.parquet"
    os.makedirs(os.path.dirname(out_path), exist_ok=True)

    labels_all = read_parquet_safe(label_path)
    labels_all["index_date"] = to_naive_datetime(labels_all["prediction_time"])
    labels_all = labels_all.dropna(subset=["person_id", "index_date"])
    labels_all["person_id"] = pd.to_numeric(labels_all["person_id"], errors="coerce").astype("Int64")
    labels_all = labels_all.dropna(subset=["person_id"])
    labels_all["person_id"] = labels_all["person_id"].astype("int64")
    all_pids = labels_all["person_id"].unique()

    writer = None
    total_rows = 0

    for i in range(0, len(all_pids), batch_size):
        batch_pids = all_pids[i:i + batch_size].tolist()
        batch_labels = labels_all[labels_all["person_id"].isin(batch_pids)][["person_id", "index_date"]].copy()
        if len(batch_pids) == 0:
            continue

        pid_str = ",".join(map(str, batch_pids))
        events_sql = f"""
            SELECT person_id, condition_concept_id AS concept_id, condition_start_date AS event_date
            FROM condition_occurrence WHERE person_id IN ({pid_str})
            UNION ALL
            SELECT person_id, drug_concept_id AS concept_id, drug_exposure_start_date AS event_date
            FROM drug_exposure WHERE person_id IN ({pid_str})
            UNION ALL
            SELECT person_id, procedure_concept_id AS concept_id, procedure_date AS event_date
            FROM procedure_occurrence WHERE person_id IN ({pid_str})
            UNION ALL
            SELECT person_id, visit_concept_id AS concept_id, visit_start_date AS event_date
            FROM visit_occurrence WHERE person_id IN ({pid_str})
        """
        events_df = pd.read_sql(events_sql, engine)
        events_df["event_date"] = to_naive_datetime(events_df["event_date"])
        events_df = events_df.dropna(subset=["person_id", "concept_id", "event_date"])
        events_df["person_id"] = pd.to_numeric(events_df["person_id"], errors="coerce").astype("Int64")
        events_df["concept_id"] = pd.to_numeric(events_df["concept_id"], errors="coerce").astype("Int64")
        events_df = events_df.dropna(subset=["person_id", "concept_id"])
        events_df["person_id"] = events_df["person_id"].astype("int64")
        events_df["concept_id"] = events_df["concept_id"].astype("int64")

        merged = events_df.merge(batch_labels, on="person_id", how="inner")
        merged["time_deltas"] = (merged["index_date"] - merged["event_date"]).dt.days
        merged = merged[merged["time_deltas"] > 0]
        merged = merged.sort_values(["person_id", "index_date", "time_deltas"], ascending=[True, True, False])

        seqs = merged.groupby(["person_id", "index_date"]).agg({"concept_id": list, "time_deltas": list}).reset_index()
        seqs = seqs.rename(columns={"concept_id": "concept_ids"})

        if not seqs.empty:
            demo_sql = f"""
                SELECT person_id, gender_concept_id, race_concept_id, ethnicity_concept_id, birth_datetime
                FROM person WHERE person_id IN ({pid_str})
            """
            demo_df = pd.read_sql(demo_sql, engine)
            demo_df["birth_datetime"] = to_naive_datetime(demo_df["birth_datetime"])
            seqs = seqs.merge(demo_df, on="person_id", how="left")

            seqs["age_at_index"] = ((seqs["index_date"] - seqs["birth_datetime"]).dt.days // 365)
            seqs["age_token"] = "AGE_" + seqs["age_at_index"].fillna(-1).astype(int).astype(str)

            def append_demo(r):
                c = r["concept_ids"]
                t = r["time_deltas"]
                demo = [
                    str(int(r["gender_concept_id"])) if pd.notna(r["gender_concept_id"]) else "UNK_GENDER",
                    str(int(r["race_concept_id"])) if pd.notna(r["race_concept_id"]) else "UNK_RACE",
                    str(int(r["ethnicity_concept_id"])) if pd.notna(r["ethnicity_concept_id"]) else "UNK_ETHNICITY",
                    r["age_token"],
                ]
                return c + demo, t + [0, 0, 0, 0]

            inj = seqs.apply(lambda x: pd.Series(append_demo(x)), axis=1)
            seqs["concept_ids"] = inj[0]
            seqs["time_deltas"] = inj[1]
            seqs = seqs.drop(columns=["gender_concept_id", "race_concept_id", "ethnicity_concept_id", "birth_datetime", "age_at_index", "age_token"])

        if seqs.empty:
            continue

        seqs["concept_ids"] = seqs["concept_ids"].apply(lambda x: [str(v) for v in x][-max_seq_len:])
        seqs["time_deltas"] = seqs["time_deltas"].apply(lambda x: x[-max_seq_len:])

        table = pa.Table.from_pandas(seqs, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(out_path, table.schema)
        writer.write_table(table)

        total_rows += len(seqs)
        print(f"[{disease_name}] EHR seq batch rows={len(seqs):,}, total={total_rows:,}")

        del events_df, merged, seqs
        gc.collect()

    if writer:
        writer.close()
    print(f"[{disease_name}] EHR seq saved: {out_path}")
    return out_path, total_rows

def extract_wearable_sequences_batched(disease_name, max_seq_len=365, batch_size=2000):
    # 兼容 wear_only / wear
    label_candidates = [
        f"{COHORT_BASE}/{disease_name}/parquet_labels/label_{disease_name}_wear_only.parquet",
        f"{COHORT_BASE}/{disease_name}/parquet_labels/label_{disease_name}_wear.parquet",
    ]
    label_path = None
    for p in label_candidates:
        if os.path.exists(p):
            label_path = p
            break
    if label_path is None:
        raise FileNotFoundError(f"No wearable label found for {disease_name}")

    out_path = f"{COHORT_BASE}/{disease_name}/inspectomop_features/transformer_{disease_name}_wear.parquet"
    os.makedirs(os.path.dirname(out_path), exist_ok=True)

    labels_all = read_parquet_safe(label_path)
    labels_all["index_date"] = to_naive_datetime(labels_all["prediction_time"])
    labels_all = labels_all.dropna(subset=["person_id", "index_date"])
    labels_all["person_id"] = pd.to_numeric(labels_all["person_id"], errors="coerce").astype("Int64")
    labels_all = labels_all.dropna(subset=["person_id"])
    labels_all["person_id"] = labels_all["person_id"].astype("int64")
    all_pids = labels_all["person_id"].unique()

    writer = None
    total_rows = 0

    for i in range(0, len(all_pids), batch_size):
        batch_pids = all_pids[i:i + batch_size].tolist()
        if not batch_pids:
            continue
        batch_labels = labels_all[labels_all["person_id"].isin(batch_pids)][["person_id", "index_date"]].copy()

        wear_df = read_parquet_safe(WEAR_PATH, filters=[("person_id", "in", batch_pids)])
        wear_df["day"] = to_naive_datetime(wear_df["day"])
        wear_df = wear_df.dropna(subset=["person_id", "day"])
        wear_df["person_id"] = pd.to_numeric(wear_df["person_id"], errors="coerce").astype("Int64")
        wear_df = wear_df.dropna(subset=["person_id"])
        wear_df["person_id"] = wear_df["person_id"].astype("int64")

        merged = wear_df.merge(batch_labels, on="person_id", how="inner")
        merged["wearable_time_deltas"] = (merged["index_date"] - merged["day"]).dt.days
        merged = merged[merged["wearable_time_deltas"] > 0]
        merged = merged.sort_values(["person_id", "index_date", "wearable_time_deltas"], ascending=[True, True, False])

        need_cols = ["wearable_time_deltas", "steps_day", "hr_avg_day", "sleep_total_day"]
        for c in need_cols:
            if c not in merged.columns:
                merged[c] = np.nan

        seqs = merged.groupby(["person_id", "index_date"]).agg({
            "wearable_time_deltas": list,
            "steps_day": list,
            "hr_avg_day": list,
            "sleep_total_day": list
        }).reset_index()

        if seqs.empty:
            continue

        pid_str = ",".join(map(str, batch_pids))
        demo_sql = f"""
            SELECT person_id, gender_concept_id, race_concept_id, ethnicity_concept_id, birth_datetime
            FROM person WHERE person_id IN ({pid_str})
        """
        demo_df = pd.read_sql(demo_sql, engine)
        demo_df["birth_datetime"] = to_naive_datetime(demo_df["birth_datetime"])
        seqs = seqs.merge(demo_df, on="person_id", how="left")

        seqs["age_at_index"] = ((seqs["index_date"] - seqs["birth_datetime"]).dt.days // 365)
        seqs["age_token"] = "AGE_" + seqs["age_at_index"].fillna(-1).astype(int).astype(str)

        def demo_pack(r):
            return [
                str(int(r["gender_concept_id"])) if pd.notna(r["gender_concept_id"]) else "UNK_GENDER",
                str(int(r["race_concept_id"])) if pd.notna(r["race_concept_id"]) else "UNK_RACE",
                str(int(r["ethnicity_concept_id"])) if pd.notna(r["ethnicity_concept_id"]) else "UNK_ETHNICITY",
                r["age_token"],
            ], [0, 0, 0, 0]

        inj = seqs.apply(lambda x: pd.Series(demo_pack(x)), axis=1)
        seqs["demo_concept_ids"] = inj[0]
        seqs["demo_time_deltas"] = inj[1]
        seqs = seqs.drop(columns=["gender_concept_id", "race_concept_id", "ethnicity_concept_id", "birth_datetime", "age_at_index", "age_token"])

        for c in ["wearable_time_deltas", "steps_day", "hr_avg_day", "sleep_total_day"]:
            seqs[c] = seqs[c].apply(lambda x: x[-max_seq_len:])

        table = pa.Table.from_pandas(seqs, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(out_path, table.schema)
        writer.write_table(table)

        total_rows += len(seqs)
        print(f"[{disease_name}] WEAR seq batch rows={len(seqs):,}, total={total_rows:,}")

        del wear_df, merged, seqs
        gc.collect()

    if writer:
        writer.close()
    print(f"[{disease_name}] WEAR seq saved: {out_path}")
    return out_path, total_rows

# =========================================================
# 5) 疾病配置 + 批处理
# =========================================================
disease_configs = {
    "atrial_fibrillation": {"additional_concept_ids": [313217], "additional_drug_concept_ids": []},
    "gad": {"additional_concept_ids": [442077, 434613, 4113821], "additional_drug_concept_ids": [739209]},
    "hyperlipidemia": {"additional_concept_ids": [432867, 432827, 438720], "additional_drug_concept_ids": []},
    "hypertension": {"additional_concept_ids": [320128, 381290, 316866], "additional_drug_concept_ids": []},
    "mdd": {"additional_concept_ids": [440383, 4282096], "additional_drug_concept_ids": [739209]},
    "gerd": {"additional_concept_ids": [4144111, 318800], "additional_drug_concept_ids": [19019418]},
    "type2_diabetes": {"additional_concept_ids": [4193704], "additional_drug_concept_ids": [1539407, 1503297]},
    "sleep_apnea": {"additional_concept_ids": [442588, 313459], "additional_drug_concept_ids": []},
    "obesity": {"additional_concept_ids": [433736, 434005], "additional_drug_concept_ids": [1539407]},
    "hf": {"additional_concept_ids": [319835], "additional_drug_concept_ids": [19075601, 19077546]},
}

results = []

for disease_name, cfg in disease_configs.items():
    print("\n" + "=" * 90)
    print(f"PROCESSING: {disease_name.upper()}")
    print("=" * 90)
    try:
        shared_pids = create_shared_patient_pool_and_splits(
            disease_name=disease_name,
            additional_concept_ids=cfg["additional_concept_ids"],
            additional_drug_concept_ids=cfg["additional_drug_concept_ids"],
            min_history_days=90
        )

        ehr_label_path, ehr_stats = create_ehr_incident_cohort(
            disease_name=disease_name,
            shared_pids=shared_pids,
            additional_concept_ids=cfg["additional_concept_ids"],
            additional_drug_concept_ids=cfg["additional_drug_concept_ids"],
            min_days=90
        )

        wear_label_path, wear_stats = create_wearable_incident_cohort(
            disease_name=disease_name,
            shared_pids=shared_pids,
            additional_concept_ids=cfg["additional_concept_ids"],
            min_days=90
        )

        ehr_seq_path, ehr_seq_rows = extract_ehr_sequences_batched(
            disease_name=disease_name, max_seq_len=512, batch_size=1000
        )
        wear_seq_path, wear_seq_rows = extract_wearable_sequences_batched(
            disease_name=disease_name, max_seq_len=365, batch_size=500
        )

        results.append({
            "Disease": disease_name,
            "Status": "Success",
            "SharedPIDs": len(shared_pids),
            "EHR_Pos": ehr_stats["positive_cases"],
            "EHR_Neg": ehr_stats["negative_cases"],
            "WEAR_Pos": wear_stats["positive_cases"],
            "WEAR_Neg": wear_stats["negative_cases"],
            "EHR_Label": ehr_label_path,
            "WEAR_Label": wear_label_path,
            "EHR_Seq_Rows": ehr_seq_rows,
            "WEAR_Seq_Rows": wear_seq_rows,
            "EHR_Seq": ehr_seq_path,
            "WEAR_Seq": wear_seq_path,
        })

    except Exception as e:
        print(f"[ERROR] {disease_name}: {e}")
        results.append({
            "Disease": disease_name,
            "Status": f"Failed: {e}",
            "SharedPIDs": 0,
            "EHR_Pos": 0,
            "EHR_Neg": 0,
            "WEAR_Pos": 0,
            "WEAR_Neg": 0,
            "EHR_Label": None,
            "WEAR_Label": None,
            "EHR_Seq_Rows": 0,
            "WEAR_Seq_Rows": 0,
            "EHR_Seq": None,
            "WEAR_Seq": None,
        })

summary_df = pd.DataFrame(results)
display(summary_df)

summary_csv = f"{REPORT_DIR}/processing_summary.csv"
summary_df.to_csv(summary_csv, index=False)
print(f"\nSaved summary: {summary_csv}")


Input files ready.
DuckDB engine created: Engine(duckdb:////home/jupyter/workspaces/controlled/omop.duckdb)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓ Loaded condition_occurrence


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓ Loaded drug_exposure


...

[hf] EHR seq batch rows=5,532, total=79,166
[hf] EHR seq batch rows=5,344, total=84,510


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[hf] EHR seq batch rows=5,743, total=90,253
[hf] EHR seq batch rows=2,472, total=92,725
[hf] EHR seq saved: /home/jupyter/workspaces/controlled/cohort_definitions/hf/inspectomop_features/transformer_hf_ehr.parquet
[hf] WEAR seq batch rows=12,292, total=12,292
[hf] WEAR seq batch rows=12,484, total=24,776
[hf] WEAR seq batch rows=12,952, total=37,728
[hf] WEAR seq batch rows=12,436, total=50,164
[hf] WEAR seq batch rows=13,596, total=63,760
[hf] WEAR seq batch rows=12,557, total=76,317
[hf] WEAR seq batch rows=12,609, total=88,926
[hf] WEAR seq batch rows=12,580, total=101,506
[hf] WEAR seq batch rows=12,580, total=114,086
[hf] WEAR seq batch rows=12,828, total=126,914
[hf] WEAR seq batch rows=12,826, total=139,740
[hf] WEAR seq batch rows=12,365, total=152,105
[hf] WEAR seq batch rows=12,814, total=164,919
[hf] WEAR seq batch rows=12,630, total=177,549
[hf] WEAR seq batch rows=12,680, total=190,229
[hf] WEAR seq batch rows=15,232, total=205,461
[hf] WEAR seq batch rows=12,708, total=21

,Disease,Status,SharedPIDs,EHR_Pos,EHR_Neg,WEAR_Pos,WEAR_Neg,EHR_Label,WEAR_Label,EHR_Seq_Rows,WEAR_Seq_Rows,EHR_Seq,WEAR_Seq
0,atrial_fibrillation,Success,21934,1386,94353,1161,552748,/home/jupyter/workspaces/controlled/cohort_def...,/home/jupyter/workspaces/controlled/cohort_def...,95739,553909,/home/jupyter/workspaces/controlled/cohort_def...,/home/jupyter/workspaces/controlled/cohort_def...
1,gad,Success,16766,8732,94263,6938,545137,/home/jupyter/workspaces/controlled/cohort_def...,/home/jupyter/workspaces/controlled/cohort_def...,102995,552075,/home/jupyter/workspaces/controlled/cohort_def...,/home/jupyter/workspaces/controlled/cohort_def...
2,hyperlipidemia,Success,14980,6411,83771,5815,499980,/home/jupyter/workspaces/controlled/cohort_def...,/home/jupyter/workspaces/controlled/cohort_def...,90182,505795,/home/jupyter/workspaces/controlled/cohort_def...,/home/jupyter/workspaces/controlled/cohort_def...
3,hypertension,Success,14945,7882,85325,6824,533667,/home/jupyter/workspaces/controlled/cohort_def...,/home/jupyter/workspaces/controlled/cohort_def...,93207,540491,/home/jupyter/workspaces/controlled/cohort_def...,/home/jupyter/workspaces/controlled/cohort_def...
4,mdd,Success,17389,3844,79259,2844,483811,/home/jupyter/workspaces/controlled/cohort_def...,/home/jupyter/workspaces/controlled/cohort_def...,83103,486655,/home/jupyter/workspaces/controlled/cohort_def...,/home/jupyter/workspaces/controlled/cohort_def...
5,gerd,Success,16236,5508,85590,4181,488261,/home/jupyter/workspaces/controlled/cohort_def...,/home/jupyter/workspaces/controlled/cohort_def...,91098,492442,/home/jupyter/workspaces/controlled/cohort_def...,/home/jupyter/workspaces/controlled/cohort_def...
6,type2_diabetes,Success,19593,3251,86454,2623,527187,/home/jupyter/workspaces/controlled/cohort_def...,/home/jupyter/workspaces/controlled/cohort_def...,89705,529810,/home/jupyter/workspaces/controlled/cohort_def...,/home/jupyter/workspaces/controlled/cohort_def...
7,sleep_apnea,Success,18544,5824,97674,5037,568234,/home/jupyter/workspaces/controlled/cohort_def...,/home/jupyter/workspaces/controlled/cohort_def...,103498,573271,/home/jupyter/workspaces/controlled/cohort_def...,/home/jupyter/workspaces/controlled/cohort_def...
8,obesity,Success,16508,5427,81315,4765,492710,/home/jupyter/workspaces/controlled/cohort_def...,/home/jupyter/workspaces/controlled/cohort_def...,86742,497475,/home/jupyter/workspaces/controlled/cohort_def...,/home/jupyter/workspaces/controlled/cohort_def...
9,hf,Success,21449,1224,91501,697,529656,/home/jupyter/workspaces/controlled/cohort_def...,/home/jupyter/workspaces/controlled/cohort_def...,92725,530353,/home/jupyter/workspaces/controlled/cohort_def...,/home/jupyter/workspaces/controlled/cohort_def...



Saved summary: /home/jupyter/workspaces/controlled/report/processing_summary.csv
